# Mapping

> The stages of a link chain that use a modulation: bits to symbols, and equalized symbols to soft bits.

In [ ]:
#| default_exp modulation.mapping

In [ ]:
#| hide
from nbdev.showdoc import *

`Mapper` and `Demapper` are a `Modulation` (see `modulation.constellation`) as stages of a link chain (see
`radio.link.chain`): `codeword` → `x`, and `x_hat`, `no_eff` → `llr`. The mapper pads the last axis to whole
symbols; the receiver drops the padding's bits.

A function of the bits per symbol instead of a modulation gives one modulation per group: e.g. the QAM of each
transmission's MCS, from its `num_bits_per_symbol` (link adaptation's, see `radio.link_adaptation`). The demapper
uses its mapper's modulation, so the two always agree.

In [ ]:
#| export
from __future__ import annotations

from typing import Callable

import numpy as np

from comm_core.block import Block
from comm_core.modulation.constellation import Modulation

In [ ]:
#| export
class Mapper(Block):
    "Bits -> symbols with a `Modulation`, the last axis padded to whole symbols."

    outputs = ('x',)

    def __init__(
        self,
        modulation: Modulation | Callable[[int], Modulation], # A modulation, or one for each number of bits per symbol
    ):
        self.modulation = modulation

    def get(
        self,
        num_bits_per_symbol: int | None = None, # A group's bits per symbol, when the modulation depends on it
    ) -> Modulation:
        "The modulation (of a group's bits per symbol, if it depends on it)."
        return self.modulation if isinstance(self.modulation, Modulation) else self.modulation(int(num_bits_per_symbol))

    def call(
        self,
        codeword, # The bits to send, (B, ..., n)
        num_bits_per_symbol: int | None = None, # The group's, for a modulation per MCS
    ):
        "The symbols, (B, ..., ceil(n / bits_per_symbol))."
        m = self.get(num_bits_per_symbol)
        pad = -codeword.shape[-1] % m.bits_per_symbol
        if pad:
            codeword = np.pad(codeword, [(0, 0)] * (codeword.ndim - 1) + [(0, pad)])
        return m.modulate(codeword)


class Demapper(Block):
    "Equalized symbols -> soft bits (LLRs), with its mapper's modulation."

    outputs = ('llr',)

    def __init__(
        self,
        mapper: Mapper, # The transmitter's mapper (its modulation)
    ):
        self.mapper = mapper

    def children(self) -> list:
        "None: the mapper is a stage of its own."
        return []

    def call(
        self,
        x_hat, # Equalized symbols
        no_eff, # Their noise power, broadcast against them
        num_bits_per_symbol: int | None = None, # The group's, for a modulation per MCS
    ):
        "log p(1) / p(0) of each bit, padding included, (B, ..., S * bits_per_symbol)."
        return self.mapper.get(num_bits_per_symbol).llr(x_hat, no_eff)

In [ ]:
from comm_core.modulation.constellation import BPSK

In [ ]:
mapper = Mapper(BPSK())
x = mapper(np.array([[0, 1, 1]]))
assert x.tolist() == [[1.0, -1.0, -1.0]] and (Demapper(mapper)(x, 0.5) < 0).tolist() == [[True, False, False]]
per_mcs = Mapper(lambda q: {1: BPSK()}[q])                               # one modulation per bits per symbol
assert per_mcs.get(1).bits_per_symbol == 1 and per_mcs(np.array([[1]]), num_bits_per_symbol=1).tolist() == [[-1.0]]

In [ ]:
from comm_core.modulation.constellation import SionnaModulation

In [ ]:
#| sionna
qam = Mapper(SionnaModulation('qam', 4))
x = qam(np.array([[1, 0, 1, 1, 0, 1]]))                                 # padded to 8 bits: 2 symbols
assert x.shape == (1, 2) and (Demapper(qam)(x, 0.01) > 0)[0, :6].tolist() == [True, False, True, True, False, True]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()